# Stokes–Brinkman and Oseen introductory methods

Compare Taylor–Hood, USFEM and Oseen stabilization with explicit affine velocity
and pressure in 2D and 3D. The operator uses grad-grad diffusion, unit viscosity
and resistance $2$. Oseen includes its declared convection vector; the pressure
has physical mean $0.7$ about the actual volume centroid.

The interface variable is the formulation's pseudotraction. It is distinct
from symmetric Cauchy traction. Errors are reported separately for velocity,
pressure and divergence. These moderate-resistance patches do not qualify the
extreme-regime Brinkman convergence campaign.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/e4fed6f12a234cc68c569d1788f14a5467876cddb5a02db07afaab6dd824ecf3/introductory_methods-companion.zip"
COMPANION_SHA256 = "e4fed6f12a234cc68c569d1788f14a5467876cddb5a02db07afaab6dd824ecf3"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("flow/introductory_methods.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)

import numpy as np


## Declare and solve Taylor–Hood Brinkman equations

The primary solve consumes these actual user-written UFL forms on native
worker-owned meshes. Positive drag $2$ leaves A locally invertible; the two
velocity translations are declared retained coarse modes, not a local kernel.
The pressure is $p=0.7+(x-1/2)+2(y-1/2)$ and the independently derived source
is $f=2u+(1,2)$. Its mean fixes the global pressure gauge.

$$
\begin{aligned}
a_T((u,p),(v,q))&=(\nabla u,\nabla v)_T+2(u,v)_T\\
 &\quad -(p,\operatorname{div}v)_T-(q,\operatorname{div}u)_T,\\
B_T\lambda&=\langle\lambda,v\rangle_{\partial T},\\
C_T(u,p)&=-\langle u,\mu\rangle_{\partial T}.
\end{aligned}
$$

The P1 Legendre trace basis uses the fixed global edge direction. The signed
multiplier is $\lambda=(-\nabla u+pI)n$ with that fixed normal; it is not
the symmetric Cauchy traction. The additional global operator is zero and
its RHS is the explicit negative Dirichlet velocity functional, followed
by zero retained-mode entries. Native execution requires compatible DOLFINx/UFL dependencies.


In [ ]:
import importlib.util
from pymhm import Equation, LocalEquations, MultiscaleProblem, assemble, columns, rows, compile_form
from pymhm.meshes.triangle import TriangleMesh
from pymhm.fem.traces.interval import FaceSpace, SkeletonSpace
from pymhm.fem.scalar.operators import boundary_data
from examples.variational_darcy import NativeFormProvider

native_available = importlib.util.find_spec("dolfinx") is not None
if native_available:
    import ufl
    from dolfinx import fem

    import basix.ufl

    mesh = TriangleMesh.unit_square()
    trace_space = SkeletonSpace(mesh, tuple(FaceSpace.uniform(1) for _ in mesh.faces), 2)
    gradient = np.array([[-1.0, 2.0], [-1.0, 1.0]])
    offset = np.array([1.0, -2.0])

    def exact_velocity(points):
        """Return the independently prescribed solenoidal affine velocity."""
        return points @ gradient.T + offset

    def exact_pressure(points):
        """Return affine pressure with physical unit-square mean 0.7."""
        return 0.7 + (points - np.array([0.5, 0.5])) @ np.array([1.0, 2.0])

    def local_forms(context):
        """Write Taylor–Hood momentum, incompressibility and trace balances."""
        element = basix.ufl.mixed_element(
            [
                basix.ufl.element("Lagrange", "triangle", 2, shape=(2,)),
                basix.ufl.element("Lagrange", "triangle", 1),
            ]
        )
        space = fem.functionspace(context.space.mesh, element)
        u, p = ufl.TrialFunctions(space)
        v, q = ufl.TestFunctions(space)
        dx = ufl.dx(domain=context.space.mesh)
        velocity_space, velocity_map = space.sub(0).collapse()
        pressure_space, pressure_map = space.sub(1).collapse()
        x = ufl.SpatialCoordinate(context.space.mesh)
        trace_basis = []
        for endpoints in context.face_endpoints:
            start, end = endpoints
            tangent = end - start
            t = sum(float(tangent[j]) * (x[j] - float(start[j])) for j in range(2))
            t /= float(tangent @ tangent)
            trace_basis.append((1, 2 * t - 1))
        exact = ufl.as_vector((1 - x[0] + 2 * x[1], -2 - x[0] + x[1]))
        force = 2 * exact + ufl.as_vector((1, 2))
        translations = np.zeros((len(velocity_map) + len(pressure_map), 2))
        translations[np.asarray(velocity_map)] = np.tile(np.eye(2), (len(velocity_map) // 2, 1))
        pressure_weights = compile_form(q * dx)
        return LocalEquations(
            (
                ufl.inner(ufl.grad(u), ufl.grad(v))
                + 2 * ufl.inner(u, v)
                - p * ufl.div(v)
                - q * ufl.div(u)
            )
            * dx,
            ufl.inner(force, v) * dx,
            columns(
                *(
                    float(sign) * basis * v[j] * context.ds(side + 1)
                    for side, sign in enumerate(context.signs)
                    for basis in trace_basis[side]
                    for j in range(2)
                )
            ),
            rows(
                *(
                    -float(sign) * basis * u[j] * context.ds(side + 1)
                    for side, sign in enumerate(context.signs)
                    for basis in trace_basis[side]
                    for j in range(2)
                )
            ),
            context.trace_dofs,
            coarse_basis=translations,
            moments=columns(v[0] * dx, v[1] * dx),
            metadata={
                "velocity_points": velocity_space.tabulate_dof_coordinates()[:, :2].copy(),
                "pressure_points": pressure_space.tabulate_dof_coordinates()[:, :2].copy(),
                "velocity_dofs": np.array(velocity_map),
                "pressure_dofs": np.array(pressure_map),
                "pressure_weights": pressure_weights,
            },
        )

    boundary, _ = boundary_data(trace_space, exact_velocity)
    problem = MultiscaleProblem(
        Equation(0, -np.r_[boundary, np.zeros(2 * len(mesh.cells))]),
        NativeFormProvider(mesh, trace_space, local_forms, 2),
        range(len(mesh.cells)),
        trace_space.size,
        (2,) * len(mesh.cells),
    )
    system = assemble(problem)
    weights = [record["pressure_weights"] for record in system.local_metadata]
    solution = system.solve(constraints=[system.mean_constraint(weights, 0.7)])
    errors, residuals = (
        {"velocity": 0.0, "pressure": 0.0},
        {"momentum": 0.0, "incompressibility": 0.0},
    )
    for response, field, record in zip(
        system.responses, solution.fields, system.local_metadata, strict=True
    ):
        errors["velocity"] = max(
            errors["velocity"],
            float(
                np.max(
                    np.abs(
                        field[record["velocity_dofs"]].reshape(-1, 2)
                        - exact_velocity(record["velocity_points"])
                    )
                )
            ),
        )
        errors["pressure"] = max(
            errors["pressure"],
            float(
                np.max(
                    np.abs(
                        field[record["pressure_dofs"]] - exact_pressure(record["pressure_points"])
                    )
                )
            ),
        )
        local = response.problem
        original_rows = (
            local.matrix @ field + local.coupling @ solution.trace[local.trace_dofs] - local.load
        )
        residuals["momentum"] = max(
            residuals["momentum"], float(np.linalg.norm(original_rows[record["velocity_dofs"]]))
        )
        residuals["incompressibility"] = max(
            residuals["incompressibility"],
            float(np.linalg.norm(original_rows[record["pressure_dofs"]])),
        )
    assert max(errors.values()) < 1e-10
    assert max(residuals.values()) < 1e-10
    print(
        {
            "native_available": True,
            "spaces": "Taylor–Hood P2/P1 / P1 vector trace",
            "maximum_nodal_errors": errors,
            "original_row_l2_by_field": residuals,
            "physical_pressure_integral": sum(
                float(w @ u) for w, u in zip(weights, solution.fields)
            ),
            "global_original_relative_residual": solution.raw_residual,
        }
    )
else:
    print(
        {
            "native_available": False,
            "status": "User-written UFL primary solve not executed: select the Pixi fem kernel.",
        }
    )


## Compare the supported methods
These comparisons invoke the existing formulation implementations through the application catalogue and preserve their numerical records. The user-defined primary UFL solve above exercises the generic local/global API; the catalogue drivers remain compatibility comparisons.


In [ ]:
from examples.tutorial_vector_variants import run_variant

selected_methods = (
    "flow-taylor-hood-2d",
    "flow-taylor-hood-3d",
    "flow-usfem-2d",
    "flow-usfem-3d",
    "flow-oseen-2d",
    "flow-oseen-3d",
)
execution_backend = "serial"  # "thread" or "process" uses importable support callables
workers = None
records = [
    run_variant(name, backend=execution_backend, workers=workers) for name in selected_methods
]
for record in records:
    print(record["variant"], record["errors"], record["provenance"])


## Check zero fields with the same formulation

In [ ]:
zero = run_variant("flow-usfem-3d", homogeneous=True)
print(zero)
